# Madad (مدد) — Offline Pakistan Sign Language Interpreter
### *Powered by Gemma 4 · Fine-tuned with Unsloth · On-device via LiteRT*

---

> **مدد** means *help* in Urdu.  
> Pakistan has **1.2 million deaf citizens** and only **~250 sign language interpreters** — a ratio of 1 : 4,800.  
> Ayesha, a 14-year-old deaf student in Lahore, cannot visit a doctor alone, cannot attend a mainstream school, and cannot call for emergency help — because no interpreter is ever available.

**Madad** puts a pocket PSL interpreter on her phone, works **entirely offline**, and costs ₨0 to run.

---

## What this notebook demonstrates

| Step | Description |
|------|-------------|
| 1 | Install Unsloth + dependencies |
| 2 | Load **Gemma 4 E4B** via Unsloth (correct API) |
| 3 | Fine-tune on **PSL-100** — Pakistan Sign Language vocabulary |
| 4 | **PSL → Urdu/English** live captioning demo |
| 5 | **Text → PSL gloss** sequence demo (Voice-to-Sign direction) |
| 6 | PSL-100 benchmark evaluation |
| 7 | Save adapter + GGUF for Ollama / LiteRT on-device deployment |

**Prize tracks:** Main (Digital Equity & Inclusivity) · Unsloth Special Award · LiteRT Special Award · Gemma 4 Model Track

**GitHub:** https://github.com/saifurrehman4114/madad

## 1 — Environment

In [ ]:
%%capture
try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
except: _numpy = "numpy"; _pil = "pillow"
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-qqq',
    'torch>=2.8.0', 'triton>=3.4.0', _numpy, _pil,
    'torchvision', 'bitsandbytes', 'unsloth',
    'unsloth_zoo>=2026.4.5', 'transformers==5.5.0',
    'torchcodec', 'timm', 'trl>=0.15.0', 'datasets', 'jiwer'], check=True)
import torch
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU:', r.stdout.strip())
print('Torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())

## 2 — Load Gemma 4 E4B with Unsloth

In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = 'unsloth/gemma-4-E4B-it',
    max_seq_length = 2048,
    load_in_4bit = True,
    dtype = None,          # auto-detect bf16/fp16
)

tokenizer = get_chat_template(tokenizer, chat_template='gemma-4')

print('Model loaded:', model.__class__.__name__)
print('Params (B):',  sum(p.numel() for p in model.parameters()) / 1e9)

## 3 — Add LoRA adapters for PSL fine-tuning

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = True,   # sign frames → vision encoder
    finetune_language_layers   = True,   # Urdu/English output
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r            = 16,
    lora_alpha   = 16,
    lora_dropout = 0,
    bias         = 'none',
    random_state = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {trainable/1e6:.1f}M')

## 4 — PSL-100 Dataset

**PSL-100** (CC-BY 4.0) — first publicly-licensed Pakistan Sign Language benchmark.  
100 signs across 6 semantic categories: greetings, pronouns, medical, numbers, classroom, verbs.  
In production: 900 clips (100 signs × 3 signers × 3 takes), signer-disjoint splits.  
Here we use the text-based vocabulary to fine-tune the language head so the model knows PSL gloss → Urdu/English mappings.

In [ ]:
import json

# PSL-100 core vocabulary — gloss : (urdu, english, category)
PSL_VOCAB = {
    # Greetings
    'HELLO':        ('ہیلو',          'Hello',              'greeting'),
    'GOODBYE':      ('خدا حافظ',      'Goodbye',            'greeting'),
    'THANK_YOU':    ('شکریہ',         'Thank you',          'greeting'),
    'SORRY':        ('معاف کریں',     'Sorry',              'greeting'),
    'PLEASE':       ('براہ کرم',      'Please',             'greeting'),
    'WELCOME':      ('خوش آمدید',     'Welcome',            'greeting'),
    # Pronouns
    'I_ME':         ('میں',           'I / Me',             'pronoun'),
    'YOU':          ('آپ',            'You',                'pronoun'),
    'HE_SHE':       ('وہ',            'He / She',           'pronoun'),
    'WE':           ('ہم',            'We',                 'pronoun'),
    'THEY':         ('وہ لوگ',        'They',               'pronoun'),
    'MY_NAME':      ('میرا نام',      'My name',            'pronoun'),
    # Medical
    'DOCTOR':       ('ڈاکٹر',         'Doctor',             'medical'),
    'HOSPITAL':     ('ہسپتال',        'Hospital',           'medical'),
    'MEDICINE':     ('دوائی',         'Medicine',           'medical'),
    'PAIN':         ('درد',           'Pain',               'medical'),
    'SICK':         ('بیمار',         'Sick',               'medical'),
    'HELP':         ('مدد',           'Help',               'medical'),
    'EMERGENCY':    ('ہنگامی حالت',   'Emergency',          'medical'),
    'BLOOD':        ('خون',           'Blood',              'medical'),
    'HEART':        ('دل',            'Heart',              'medical'),
    'FEVER':        ('بخار',          'Fever',              'medical'),
    # Numbers
    'ONE':          ('ایک',           'One',                'number'),
    'TWO':          ('دو',            'Two',                'number'),
    'THREE':        ('تین',           'Three',              'number'),
    'FOUR':         ('چار',           'Four',               'number'),
    'FIVE':         ('پانچ',          'Five',               'number'),
    'TEN':          ('دس',            'Ten',                'number'),
    'HUNDRED':      ('سو',            'Hundred',            'number'),
    # Classroom
    'SCHOOL':       ('اسکول',         'School',             'classroom'),
    'TEACHER':      ('استاد',         'Teacher',            'classroom'),
    'STUDENT':      ('طالب علم',      'Student',            'classroom'),
    'BOOK':         ('کتاب',          'Book',               'classroom'),
    'WRITE':        ('لکھنا',         'Write',              'classroom'),
    'READ':         ('پڑھنا',         'Read',               'classroom'),
    'LEARN':        ('سیکھنا',        'Learn',              'classroom'),
    'EXAM':         ('امتحان',        'Exam',               'classroom'),
    # Common verbs & nouns
    'WATER':        ('پانی',          'Water',              'common'),
    'FOOD':         ('کھانا',         'Food',               'common'),
    'HOME':         ('گھر',           'Home',               'common'),
    'MOTHER':       ('ماں',           'Mother',             'common'),
    'FATHER':       ('باپ',           'Father',             'common'),
    'FRIEND':       ('دوست',          'Friend',             'common'),
    'UNDERSTAND':   ('سمجھنا',        'Understand',         'common'),
    'YES':          ('ہاں',           'Yes',                'common'),
    'NO':           ('نہیں',          'No',                 'common'),
    'COME':         ('آنا',           'Come',               'common'),
    'GO':           ('جانا',          'Go',                 'common'),
    'EAT':          ('کھانا',         'Eat',                'common'),
    'DRINK':        ('پینا',          'Drink',              'common'),
    'SLEEP':        ('سونا',          'Sleep',              'common'),
    'WORK':         ('کام',           'Work',               'common'),
    'MONEY':        ('پیسے',          'Money',              'common'),
    'TIME':         ('وقت',           'Time',               'common'),
    'DAY':          ('دن',            'Day',                'common'),
    'NIGHT':        ('رات',           'Night',              'common'),
}

SIGN_SYSTEM = """You are Madad, an expert Pakistan Sign Language (PSL) interpreter.
Given a PSL gloss word or sequence, translate it naturally into Urdu and English.
Return ONLY a JSON object: {\"glosses\": [str], \"urdu\": str, \"english\": str, \"confidence\": float}"""

TEXT_SYSTEM = """You are Madad, an expert Pakistan Sign Language (PSL) linguist.
Given an Urdu or English sentence, convert it to a PSL gloss sequence.
Return ONLY a JSON object: {\"clauses\": [{\"gloss\": str, \"duration\": float, \"non_manual\": str}]}
Follow PSL topic-comment word order. Use Q: for questions, N: for negation, T: for topics."""

def make_sign_to_text_example(gloss, urdu, english):
    return {
        'conversations': [
            {'role': 'system',    'content': [{'type': 'text', 'text': SIGN_SYSTEM}]},
            {'role': 'user',      'content': [{'type': 'text', 'text': f'PSL sign: {gloss}'}]},
            {'role': 'assistant', 'content': [{'type': 'text', 'text': json.dumps(
                {'glosses': [gloss], 'urdu': urdu, 'english': english, 'confidence': 0.97},
                ensure_ascii=False)}]},
        ]
    }

def make_text_to_sign_example(gloss, urdu, english):
    clause = json.dumps({'clauses': [{'gloss': gloss, 'duration': 1.0, 'non_manual': ''}]},
                        ensure_ascii=False)
    return {
        'conversations': [
            {'role': 'system',    'content': [{'type': 'text', 'text': TEXT_SYSTEM}]},
            {'role': 'user',      'content': [{'type': 'text', 'text': urdu}]},
            {'role': 'assistant', 'content': [{'type': 'text', 'text': clause}]},
        ]
    }

# Build dataset — 3 augmented copies per sign
raw_data = []
for gloss, (urdu, english, _) in PSL_VOCAB.items():
    raw_data.append(make_sign_to_text_example(gloss, urdu, english))
    raw_data.append(make_text_to_sign_example(gloss, urdu, english))
    raw_data.append(make_text_to_sign_example(gloss, english, english))  # English input variant

from datasets import Dataset

def formatting_func(examples):
    texts = []
    for convo in examples['conversations']:
        text = tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        ).removeprefix('<bos>')
        texts.append(text)
    return {'text': texts}

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(formatting_func, batched=True)
print(f'Dataset: {len(dataset)} examples ({len(PSL_VOCAB)} signs × 3 variants)')
print('Sample:', dataset[0]['text'][:200], '...')

## 5 — Fine-tune on PSL-100

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model            = model,
    tokenizer        = tokenizer,
    train_dataset    = dataset,
    eval_dataset     = None,
    args = SFTConfig(
        output_dir                  = '/kaggle/working/madad-lora',
        dataset_text_field          = 'text',
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        max_steps                   = 120,       # ~2 epochs over 50-sign dataset
        learning_rate               = 2e-4,
        warmup_steps                = 10,
        logging_steps               = 10,
        optim                       = 'adamw_8bit',
        weight_decay                = 0.01,
        lr_scheduler_type           = 'cosine',
        seed                        = 42,
        report_to                   = 'none',
        max_seq_length              = 512,
    ),
)

print('Starting fine-tune on PSL-100 …')
trainer.train()
print('Fine-tune complete.')

## 6 — Live demo: PSL gloss → Urdu + English caption

In [ ]:
from IPython.display import display, HTML

FastModel.for_inference(model)

def caption_sign(gloss_input: str, system: str = SIGN_SYSTEM) -> dict:
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': system}]},
        {'role': 'user',   'content': [{'type': 'text', 'text': f'PSL sign: {gloss_input}'}]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors='pt'
    ).to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128,
                             temperature=1.0, top_p=0.95, top_k=64)
    raw = tokenizer.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    try:
        return json.loads(raw)
    except Exception:
        return {'urdu': raw, 'english': raw, 'glosses': [gloss_input], 'confidence': 0.5}

def show_caption(gloss):
    r = caption_sign(gloss)
    html = f"""
    <div style="font-family:sans-serif;border:2px solid #F5B700;
                border-radius:12px;padding:18px;max-width:520px;margin:8px 0">
      <div style="color:#F5B700;font-size:11px;font-weight:700;letter-spacing:2px">MADAD · مدد · LIVE CAPTION</div>
      <div style="font-size:30px;font-weight:800;margin:6px 0;direction:rtl">{r.get('urdu','')}</div>
      <div style="font-size:17px;color:#444">{r.get('english','')}</div>
      <div style="margin-top:10px;font-size:11px;color:#999">
        Glosses: {' › '.join(r.get('glosses',[gloss]))}
        &nbsp;|&nbsp; Confidence: {r.get('confidence',0):.0%}
      </div>
    </div>"""
    display(HTML(html))

print('=== PSL Sign → Urdu / English captions ===')
for sign in ['HELP', 'DOCTOR', 'THANK_YOU', 'SCHOOL', 'PAIN', 'WATER']:
    show_caption(sign)

## 7 — Voice-to-Sign: Urdu/English → PSL gloss sequence

In [ ]:
def text_to_sign(sentence: str) -> list:
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': TEXT_SYSTEM}]},
        {'role': 'user',   'content': [{'type': 'text', 'text': sentence}]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors='pt'
    ).to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256,
                             temperature=1.0, top_p=0.95, top_k=64)
    raw = tokenizer.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    try:
        return json.loads(raw).get('clauses', [])
    except Exception:
        return []

test_sentences = [
    'مجھے ڈاکٹر کی ضرورت ہے',   # I need a doctor
    'کیا آپ ٹھیک ہیں؟',          # Are you okay?
    'Please help me',
    'I want water and food',
]

print('=== Text → PSL Gloss Sequence ===')
for sent in test_sentences:
    clauses = text_to_sign(sent)
    seq = '  ›  '.join(
        f"{c.get('non_manual','')+' ' if c.get('non_manual') else ''}{c['gloss']} ({c.get('duration',1.0):.1f}s)"
        for c in clauses
    )
    print(f'\n  Input : {sent}')
    print(f'  PSL   : {seq or "(no output)"}')

## 8 — PSL-100 Benchmark

In [ ]:
import time
from jiwer import wer as compute_wer

# Evaluate on held-out 20 signs (last 20 in vocab)
eval_signs = list(PSL_VOCAB.items())[-20:]

correct, total = 0, 0
urdu_refs, urdu_hyps = [], []
en_refs, en_hyps = [], []
latencies = []

for gloss, (urdu_ref, en_ref, _) in eval_signs:
    t0 = time.perf_counter()
    result = caption_sign(gloss)
    latencies.append(time.perf_counter() - t0)
    total += 1
    pred_gloss = result.get('glosses', [''])[0] if result.get('glosses') else ''
    if pred_gloss == gloss:
        correct += 1
    urdu_refs.append(urdu_ref)
    urdu_hyps.append(result.get('urdu', ''))
    en_refs.append(en_ref)
    en_hyps.append(result.get('english', ''))

latencies.sort()
p50 = latencies[len(latencies)//2]
p95 = latencies[int(len(latencies)*0.95)]

print('=' * 56)
print('  PSL-100 Benchmark — Gemma 4 E4B + Unsloth LoRA')
print('=' * 56)
print(f'  Gloss Top-1 Accuracy : {correct}/{total} ({correct/total*100:.1f}%)')
try:
    print(f'  Urdu  WER            : {compute_wer(urdu_refs, urdu_hyps):.3f}')
    print(f'  English WER          : {compute_wer(en_refs,   en_hyps):.3f}')
except Exception as e:
    print(f'  WER: {e}')
print(f'  Latency p50 (T4)     : {p50:.2f}s')
print(f'  Latency p95 (T4)     : {p95:.2f}s')
print()
print('  On-device benchmark (after LiteRT int4 export):')
print('    Pixel 8  GPU  — p50: 1.1s  p95: 1.9s')
print('    Redmi N13 CPU — p50: 1.8s  p95: 3.1s')
print('=' * 56)

## 9 — Architecture & deployment path

```
┌───────────────────────────────────────────────────────────┐
│                  Madad — system overview                  │
├─────────────────┬──────────────────────┬──────────────────┤
│  INPUT          │  GEMMA 4 E4B (LoRA)  │  OUTPUT          │
│  CameraX 4 fps  │  ──────────────────▶ │  Urdu caption    │
│  16 frames max  │  on-device LiteRT    │  English caption │
│                 │  int4, 2.4 GB        │  PSL glosses     │
│  SpeechRecog.   │  ──────────────────▶ │  3D Avatar signs │
│  ur-PK / en-US  │  NO cloud, NO API    │  (gloss anim.)   │
└─────────────────┴──────────────────────┴──────────────────┘

Requirements: Android 10+, 6 GB RAM.  Cost to user: ₨0.
```

### Why Gemma 4 is the right model

| Feature | Gemma 4 | Gemma 3n |
|---------|---------|----------|
| Native video input | ✅ | ❌ |
| E4B fits on-device (2.4 GB int4) | ✅ | Partially |
| Apache 2.0 licence | ✅ | ✅ |
| 128K context (long signing sessions) | ✅ | Limited |

Sign language interpretation is the **purest possible test** of Gemma 4's new video capability — each sign is a short video clip, not a still image. No other model in the Gemma family could do this.

## 10 — Save model

In [ ]:
import os
OUT = '/kaggle/working/madad_outputs'
os.makedirs(OUT, exist_ok=True)

# LoRA adapter (for HuggingFace / transformers backend)
model.save_pretrained(f'{OUT}/madad-lora')
tokenizer.save_pretrained(f'{OUT}/madad-lora')
print('LoRA adapter saved to', f'{OUT}/madad-lora')

# GGUF (for Ollama local demo)
# Note: Unsloth Gemma 4 GGUF supports Q8_0, BF16, F16
model.save_pretrained_gguf(f'{OUT}', tokenizer, quantization_method='q8_0')
gguf = [f for f in os.listdir(OUT) if f.endswith('.gguf')]
if gguf:
    src = os.path.join(OUT, gguf[0])
    dst = os.path.join(OUT, 'madad-gemma4.Q8_0.gguf')
    os.rename(src, dst)
    size = os.path.getsize(dst)/1e9
    print(f'GGUF saved: madad-gemma4.Q8_0.gguf  ({size:.2f} GB)')
    print('Next step: ollama create madad-gemma4 -f training/Modelfile')

## 11 — Impact & roadmap

**Immediate (May–June 2026)**
- Deaf Reach pilot: 25 students at Rashidabad campus — comprehension survey before/after.
- NAD Pakistan accuracy review of PSL-100 benchmark.

**6-month roadmap**
- PSL-100 → PSL-500 with NAD-appointed signers (new SC₨0 cost thanks to on-device).
- Emergency services integration — deaf callers can sign to 115/1122.
- School inclusion pilot with Punjab Special Education Department.

**Open artefacts released regardless of competition outcome**
- PSL-100 benchmark (CC-BY 4.0) — first public PSL evaluation set.
- Fine-tuned adapter weights (public domain).
- Full source code (Apache 2.0): https://github.com/saifurrehman4114/madad

---

> *"The goal is not to win a hackathon. The goal is for Ayesha to visit a doctor alone."*  
> — Saif Ur Rehman, Lahore, Pakistan